# Analisis Menentukan Jumlah Cluster pada K-Means Clustering

**Mata Kuliah:** Proyek Sains Data — Semester 5
**Data:** Fitur TSFEL hasil ekstraksi NO₂, CO, dan SO₂ (37 kecamatan, 68 fitur per polutan)

## Tujuan Notebook

Notebook ini menjawab pertanyaan: **berapa jumlah cluster (K) yang paling baik** untuk
mengelompokkan 37 kecamatan berdasarkan karakteristik kualitas udaranya, menggunakan
algoritma **K-Means Clustering**.

Analisis dilakukan dalam dua skenario:

1. **Data gabungan (204 fitur)** — menggabungkan 68 fitur NO₂ + 68 fitur CO + 68 fitur
   SO₂ per kecamatan, direduksi dimensinya menjadi **37 komponen PCA**, lalu ditentukan
   jumlah cluster terbaik.
2. **Data per-polutan (68 fitur)** — prosedur yang sama (standardisasi → PCA → analisis
   Elbow & Silhouette Score) diulang secara terpisah untuk NO₂, CO, dan SO₂.

Dua metode dipakai untuk menentukan K terbaik:
- **Elbow Method** — melihat titik "siku" pada grafik *within-cluster sum of squares*
  (WCSS/inertia) terhadap jumlah cluster.
- **Silhouette Score** — mengukur seberapa baik setiap titik data cocok dengan
  cluster-nya sendiri dibanding cluster lain, dengan nilai skor dari −1 hingga 1
  (semakin tinggi semakin baik).

## Import Pustaka

In [6]:
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

DATA_DIR = "./features"  # TODO: sesuaikan jika file CSV disimpan di folder lain, mis. "../data/features/"

FILES = {
    "NO2": "./features/ekstraksi_fitur_no2.csv",
    "CO": "./features/ekstraksi_fitur_co.csv",
    "SO2": "./features/ekstraksi_fitur_so2.csv",
}

## Memuat & Menggabungkan Data (Kunci: `nama`, BUKAN `id`)

> ⚠️ **Penting:** kolom `id` pada ketiga file **tidak konsisten** — `id` yang sama bisa
> merujuk ke orang/kecamatan yang berbeda di tiap file (14 dari 37 baris mismatch bila
> digabung pakai `id`). Kolom `nama` terbukti konsisten (kecuali satu perbedaan spasi
> minor), sehingga dipakai sebagai kunci penggabungan di sini — dinormalisasi dulu
> (hilangkan spasi ganda, huruf kecil semua) supaya perbedaan format kecil tidak
> menyebabkan baris gagal tergabung.

In [7]:
def normalize_name(name):
    """Menyeragamkan format nama: huruf kecil semua, hapus tanda titik,
    dan rapikan spasi berlebih -- supaya perbedaan format kecil
    (mis. 'A.Choiril' vs 'A. Choiril') tetap dianggap sama."""
    name = str(name).strip().lower()
    name = name.replace(".", " ")
    name = re.sub(r"\s+", " ", name)
    return name.strip()


raw_data = {}
for pollutant, filename in FILES.items():
    df = pd.read_csv(DATA_DIR + filename)
    df["nama_key"] = df["nama"].apply(normalize_name)
    raw_data[pollutant] = df
    print(f"{pollutant}: {df.shape[0]} baris, {df.shape[1]} kolom")

# Verifikasi: pastikan ketiga file benar-benar punya 37 nama yang sama persis
keys_no2 = set(raw_data["NO2"]["nama_key"])
keys_co = set(raw_data["CO"]["nama_key"])
keys_so2 = set(raw_data["SO2"]["nama_key"])
print(f"\nJumlah nama unik -> NO2: {len(keys_no2)} | CO: {len(keys_co)} | SO2: {len(keys_so2)}")
print(f"Ketiga file punya nama yang identik? {keys_no2 == keys_co == keys_so2}")
if not (keys_no2 == keys_co == keys_so2):
    print("Nama yang tidak konsisten:", (keys_no2 | keys_co | keys_so2) - (keys_no2 & keys_co & keys_so2))

FileNotFoundError: [Errno 2] No such file or directory: './features./features/ekstraksi_fitur_no2.csv'

In [ ]:
feature_cols = [c for c in raw_data["NO2"].columns if c not in ("id", "nama", "daerah", "nama_key")]
print(f"Jumlah kolom fitur per polutan: {len(feature_cols)}")

# Gabungkan ketiga dataframe berdasarkan 'nama_key', beri prefix nama polutan pada tiap fitur
merged = raw_data["NO2"][["nama_key", "nama", "daerah"] + feature_cols].rename(
    columns={c: f"NO2_{c}" for c in feature_cols}
)

for pollutant in ["CO", "SO2"]:
    df_p = raw_data[pollutant][["nama_key"] + feature_cols].rename(
        columns={c: f"{pollutant}_{c}" for c in feature_cols}
    )
    merged = merged.merge(df_p, on="nama_key", how="inner")

print(f"\nHasil gabungan: {merged.shape[0]} baris, {merged.shape[1]} kolom")
print(f"(Diharapkan: 37 baris, 3 metadata + {len(feature_cols)}x3 = {3 + len(feature_cols)*3} kolom)")

if merged.shape[0] != 37:
    print("\n[PERINGATAN] Jumlah baris hasil gabungan tidak 37 — ada nama yang tidak "
          "cocok di ketiga file. Cek nama yang hilang di bawah.")
    all_keys = set(raw_data["NO2"]["nama_key"]) | set(raw_data["CO"]["nama_key"]) | set(raw_data["SO2"]["nama_key"])
    common_keys = set(raw_data["NO2"]["nama_key"]) & set(raw_data["CO"]["nama_key"]) & set(raw_data["SO2"]["nama_key"])
    print("Nama yang TIDAK ada di ketiga file:", all_keys - common_keys)

merged.head()

In [ ]:
# Matriks fitur gabungan (204 kolom): metadata dipisah dari fitur numerik
meta_cols = ["nama", "daerah"]
feature_cols_combined = [c for c in merged.columns if c not in ("nama_key", "nama", "daerah")]

X_combined = merged[feature_cols_combined].apply(pd.to_numeric, errors="coerce")
print(f"Matriks fitur gabungan: {X_combined.shape[0]} baris x {X_combined.shape[1]} fitur")
print(f"Missing value pada matriks gabungan: {X_combined.isna().sum().sum()}")

---
# Bagian 1 — Data Gabungan (204 Fitur → PCA 37 Komponen)

## 1.1 Standardisasi Fitur

Sebelum PCA dan K-Means, seluruh fitur **wajib distandardisasi** (mean = 0, std = 1)
menggunakan `StandardScaler`. Ini penting karena:
- 68 fitur TSFEL punya **satuan dan skala yang sangat berbeda** (mis. `calc_mean`
  bersatuan konsentrasi polutan, sedangkan `zero_cross` cuma berupa hitungan/count).
- Tanpa standardisasi, fitur dengan skala nilai lebih besar akan **mendominasi**
  perhitungan jarak Euclidean pada PCA maupun K-Means, membuat fitur berskala kecil
  jadi terabaikan meskipun sebenarnya informatif.

In [ ]:
scaler_combined = StandardScaler()
X_combined_scaled = scaler_combined.fit_transform(X_combined)

print(f"Data terstandardisasi: {X_combined_scaled.shape[0]} baris x {X_combined_scaled.shape[1]} fitur")
print(f"Rata-rata tiap fitur setelah standardisasi (harus ≈0): {X_combined_scaled.mean():.6f}")
print(f"Std tiap fitur setelah standardisasi (harus ≈1)     : {X_combined_scaled.std():.6f}")

## 1.2 Reduksi Dimensi — PCA ke 37 Komponen

Dengan 37 sampel (kecamatan) dan 204 fitur, jumlah fitur **jauh lebih banyak** dari
jumlah data — situasi yang disebut **curse of dimensionality**, di mana jarak antar
titik data menjadi kurang bermakna dan clustering jadi tidak stabil.

**PCA (Principal Component Analysis)** meringkas 204 fitur menjadi kombinasi linear
baru (komponen utama) yang mempertahankan variansi data sebanyak mungkin. Jumlah
komponen maksimum yang valid secara matematis adalah
$\min(n_{\text{sampel}}, n_{\text{fitur}}) = \min(37, 204) = 37$ — sesuai permintaan
tugas, PCA direduksi ke **37 komponen** (PCA1 sampai PCA37).

In [ ]:
N_COMPONENTS = min(37, X_combined_scaled.shape[0], X_combined_scaled.shape[1])

pca_combined = PCA(n_components=N_COMPONENTS, random_state=42)
X_combined_pca = pca_combined.fit_transform(X_combined_scaled)

print(f"Data setelah PCA: {X_combined_pca.shape[0]} baris x {X_combined_pca.shape[1]} komponen")
print(f"Total variansi yang dijelaskan oleh {N_COMPONENTS} komponen: "
      f"{pca_combined.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
# Visualisasi variansi yang dijelaskan tiap komponen (scree plot) + kumulatif
cumulative_variance = np.cumsum(pca_combined.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(1, N_COMPONENTS + 1), pca_combined.explained_variance_ratio_ * 100, color="steelblue")
axes[0].set_title("Variansi Dijelaskan per Komponen PCA (Data Gabungan)")
axes[0].set_xlabel("Komponen PCA")
axes[0].set_ylabel("Variansi Dijelaskan (%)")

axes[1].plot(range(1, N_COMPONENTS + 1), cumulative_variance * 100, marker="o", color="darkorange")
axes[1].axhline(90, color="gray", linestyle="--", label="90%")
axes[1].set_title("Variansi Kumulatif (Data Gabungan)")
axes[1].set_xlabel("Jumlah Komponen PCA")
axes[1].set_ylabel("Variansi Kumulatif (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

n_for_90 = int(np.argmax(cumulative_variance >= 0.90) + 1)
print(f"Jumlah komponen PCA yang dibutuhkan untuk menjelaskan 90% variansi: {n_for_90}")

## 1.3 Menentukan Jumlah Cluster — Elbow Method

**Elbow Method** menjalankan K-Means untuk berbagai nilai K, lalu memplot **inertia**
(*within-cluster sum of squares*, WCSS) terhadap K:

$$\text{Inertia} = \sum_{i=1}^{n} \min_{\mu_j \in C} \|x_i - \mu_j\|^2$$

di mana $x_i$ adalah titik data dan $\mu_j$ adalah centroid cluster terdekat. Inertia
selalu **menurun** seiring bertambahnya K (karena setiap titik lebih mudah didekati
centroidnya sendiri), tapi penurunannya **melambat** setelah titik tertentu — titik
"siku" (*elbow*) inilah yang menunjukkan jumlah cluster optimal, di mana penambahan
cluster berikutnya tidak lagi memberi perbaikan signifikan.

In [ ]:
def compute_elbow_silhouette(X, k_range):
    """Menghitung inertia (elbow) dan silhouette score untuk setiap K dalam k_range."""
    inertias = []
    silhouettes = []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        inertias.append(kmeans.inertia_)
        silhouettes.append(silhouette_score(X, labels))
    return inertias, silhouettes


K_RANGE = range(2, 11)  # jumlah sampel cuma 37, K diuji dari 2 s.d. 10

inertias_combined, silhouettes_combined = compute_elbow_silhouette(X_combined_pca, K_RANGE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(list(K_RANGE), inertias_combined, marker="o", color="steelblue")
axes[0].set_title("Elbow Method — Data Gabungan (PCA 37 Komponen)")
axes[0].set_xlabel("Jumlah Cluster (K)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_RANGE), silhouettes_combined, marker="o", color="darkorange")
axes[1].set_title("Silhouette Score — Data Gabungan (PCA 37 Komponen)")
axes[1].set_xlabel("Jumlah Cluster (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.4 Silhouette Score

**Silhouette Score** untuk satu titik data $i$ dihitung sebagai:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$$

- $a(i)$ = rata-rata jarak titik $i$ ke semua titik lain **dalam cluster yang sama**
  (mengukur seberapa rapat cluster-nya sendiri)
- $b(i)$ = rata-rata jarak titik $i$ ke semua titik di **cluster terdekat lainnya**
  (mengukur seberapa jauh dari cluster tetangga)

Nilai $s(i)$ berkisar dari **−1 hingga 1**:
- Mendekati **+1** → titik data cocok dengan cluster-nya, jauh dari cluster lain (baik)
- Mendekati **0** → titik data berada di perbatasan antar dua cluster
- Negatif → titik data kemungkinan salah cluster

Skor keseluruhan adalah **rata-rata $s(i)$ dari semua titik data**. K dengan silhouette
score tertinggi dianggap sebagai jumlah cluster terbaik.

In [ ]:
best_k_silhouette_combined = list(K_RANGE)[int(np.argmax(silhouettes_combined))]
best_score_combined = max(silhouettes_combined)

print("=== Ringkasan Data Gabungan (PCA 37 Komponen) ===")
for k, inertia, sil in zip(K_RANGE, inertias_combined, silhouettes_combined):
    marker = "  <-- TERBAIK" if k == best_k_silhouette_combined else ""
    print(f"K={k}: inertia={inertia:,.2f} | silhouette={sil:.4f}{marker}")

print(f"\n✅ Jumlah cluster terbaik (berdasarkan Silhouette Score tertinggi): "
      f"K = {best_k_silhouette_combined} (skor = {best_score_combined:.4f})")
print("   Untuk Elbow Method, lihat grafik di atas dan identifikasi titik 'siku' secara visual —")
print("   titik di mana penurunan inertia mulai melandai.")

---
# Bagian 2 — Data Per-Polutan (68 Fitur, Diulang untuk NO₂, CO, SO₂)

Prosedur yang **sama persis** (standardisasi → PCA → Elbow Method → Silhouette Score)
diulang secara terpisah untuk data 68 fitur masing-masing polutan — dibungkus sebagai
fungsi `analyze_pollutant()` agar tidak menulis ulang kode tiga kali, lalu dijalankan
dalam loop untuk NO₂, CO, dan SO₂.

In [ ]:
def analyze_pollutant(df, feature_cols, pollutant_name, k_range=range(2, 11)):
    """Standardisasi -> PCA (37 komponen) -> Elbow & Silhouette untuk satu polutan."""
    X = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_components = min(37, X_scaled.shape[0], X_scaled.shape[1])
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    print(f"\n=== {pollutant_name} — PCA {n_components} komponen "
          f"(variansi dijelaskan: {pca.explained_variance_ratio_.sum()*100:.2f}%) ===")

    inertias, silhouettes = compute_elbow_silhouette(X_pca, k_range)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(list(k_range), inertias, marker="o", color="steelblue")
    axes[0].set_title(f"Elbow Method — {pollutant_name} (68 Fitur, PCA {n_components})")
    axes[0].set_xlabel("Jumlah Cluster (K)")
    axes[0].set_ylabel("Inertia (WCSS)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(list(k_range), silhouettes, marker="o", color="darkorange")
    axes[1].set_title(f"Silhouette Score — {pollutant_name} (68 Fitur, PCA {n_components})")
    axes[1].set_xlabel("Jumlah Cluster (K)")
    axes[1].set_ylabel("Silhouette Score")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    best_k = list(k_range)[int(np.argmax(silhouettes))]
    best_score = max(silhouettes)
    print(f"✅ {pollutant_name}: K terbaik = {best_k} (silhouette score = {best_score:.4f})")

    return {"pollutant": pollutant_name, "best_k": best_k, "best_silhouette": best_score,
            "inertias": inertias, "silhouettes": silhouettes}


results_per_pollutant = []
for pollutant in ["NO2", "CO", "SO2"]:
    result = analyze_pollutant(raw_data[pollutant], feature_cols, pollutant)
    results_per_pollutant.append(result)

---
# Ringkasan & Kesimpulan

Tabel berikut merangkum jumlah cluster terbaik (berdasarkan Silhouette Score
tertinggi) untuk data gabungan (204 fitur → PCA 37 komponen) dan masing-masing
polutan (68 fitur → PCA 37 komponen).

In [ ]:
summary_rows = [{
    "Data": "Gabungan (NO2+CO+SO2, 204 fitur -> PCA 37)",
    "K Terbaik": best_k_silhouette_combined,
    "Silhouette Score": round(best_score_combined, 4),
}]

for r in results_per_pollutant:
    summary_rows.append({
        "Data": f"{r['pollutant']} (68 fitur -> PCA 37)",
        "K Terbaik": r["best_k"],
        "Silhouette Score": round(r["best_silhouette"], 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## Cara Membaca Hasil

- **K terbaik berdasarkan Silhouette Score** dipilih otomatis (skor tertinggi) pada
  tabel di atas — ini jawaban kuantitatif untuk "berapa baiknya data ini di-cluster".
- **Elbow Method** bersifat lebih subjektif (dibaca dari grafik, mencari titik siku)
  — gunakan sebagai **pembanding**: kalau K hasil Elbow dan K hasil Silhouette Score
  cocok/berdekatan, itu memperkuat keyakinan bahwa K tersebut memang jumlah cluster
  yang tepat.
- Jika K terbaik pada data gabungan **berbeda** dari K terbaik di masing-masing
  polutan, itu wajar — artinya pola pengelompokan kecamatan berdasarkan "kualitas
  udara keseluruhan" tidak selalu sama dengan pola pengelompokan berdasarkan satu
  polutan saja. Untuk laporan akhir, K dari **data gabungan** biasanya yang paling
  relevan dipakai sebagai jumlah cluster final, karena merepresentasikan kombinasi
  ketiga polutan sekaligus.